# PerGAZE GazeformerISP-S: k-shot subject embedding adaptation

Enable Internet, select GPU T4 x2 (one GPU also works), and attach:
1. PerGAZE dataset containing support.json, test_unseen.json, images and attention_reasoning.
2. Output of the pretrained Gazeformer run containing checkpoints/best.pth.

K is a hyperparameter. Default K=5, seed=10, epochs=3. Exactly K samples per
support observer are selected with a seeded sampler. Only the new subject
embedding rows are trained; all other weights, old observer rows and buffers
stay frozen. Vocabulary, image geometry and max_length are inherited from the
base checkpoint. Best adapted epoch is selected on test_unseen as requested;
this makes it a model-selection set, not an untouched test estimate.


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/kaggle/working/IndividualScanpath")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ntnghia06/IndividualScanpath.git", str(CODE)], check=True)
SRC = CODE / "PerGAZE/GazeformerISP-S/src"
assert (SRC / "train.py").is_file(), "Update the repository for GazeformerISP-S"


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze sentence-transformers


In [ ]:
import torch
import torchvision

assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings"
GPU_IDS = list(range(min(2, torch.cuda.device_count())))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
for i in GPU_IDS:
    print(i, torch.cuda.get_device_name(i))


Choose k and seed, then select the correct base checkpoint. Automatic selection works when there is exactly one best.pth in Input; otherwise set CHECKPOINT manually using one of the printed paths.


In [ ]:
K = 5
SEED = 10
EPOCHS = 3
DATA = Path("/kaggle/input/datasets/ngtrongnghia/pergaze/dataset")
CHECKPOINT = None  # Or Path("/kaggle/input/<base-output>/<run>/checkpoints/best.pth")
TEXT_EMBEDDINGS = None  # Set the NPZ from the base cached-v3 training output.
FEATURE_DIR = Path("/kaggle/working/gazeformer_image_features_fp32_v2")

candidates = sorted(Path("/kaggle/input").rglob("best.pth"))
for i, candidate in enumerate(candidates):
    print(i, candidate)
if CHECKPOINT is None:
    if len(candidates) != 1:
        raise ValueError("Set CHECKPOINT to the base Gazeformer best.pth from the list above")
    CHECKPOINT = candidates[0]
assert CHECKPOINT.is_file()
for name in ("support.json", "test_unseen.json", "images"):
    assert (DATA / name).exists(), f"Missing: {DATA / name}"
RUN = Path(f"/kaggle/working/pergaze_gazeformer_s_k{K}_seed{SEED}_by_condition")
COMMON = [
    "--checkpoint", str(CHECKPOINT),
    "--support_file", str(DATA / "support.json"),
    "--test_file", str(DATA / "test_unseen.json"),
    "--img_dir", str(DATA / "images"),
    "--att_dir", str(DATA / "attention_reasoning"),
    "--k", str(K), "--seed", str(SEED), "--epoch", str(EPOCHS),
    "--device", "cuda:0", "--gpu_ids", *map(str, GPU_IDS),
]
if TEXT_EMBEDDINGS is None:
    archives = sorted(Path("/kaggle/input").rglob("gazeformer_task_embeddings.npz"))
    if len(archives) != 1:
        raise ValueError("Set TEXT_EMBEDDINGS to the original NPZ from the base run")
    TEXT_EMBEDDINGS = archives[0]
# Precompute support/unseen image features, or point FEATURE_DIR to an imported cache.
subprocess.run([
    sys.executable, "-u", str(CODE / "PerGAZE/GazeformerISP/src/preprocess/feature_extractor.py"),
    "--mode", "images", "--files", str(DATA / "support.json"), str(DATA / "test_unseen.json"),
    "--img_dir", str(DATA / "images"), "--feature_dir", str(FEATURE_DIR),
    "--device", "cuda:0", "--gpu_ids", *map(str, GPU_IDS), "--batch", "4", "--workers", "2",
], check=True)
COMMON += ["--text_embeddings", str(TEXT_EMBEDDINGS), "--feature_dir", str(FEATURE_DIR)]
def run_script(script, *arguments):
    subprocess.run([sys.executable, "-u", str(SRC / script), *COMMON, *arguments], check=True)


Finetune all selected support samples for 3 epochs. Each epoch evaluates the complete unseen set and updates best.pth if the ScanMatch harmonic mean improves. Evaluation draws use the same seed for fair epoch comparisons. Both training and evaluation have progress bars.


In [ ]:
run_script("train.py", "--log_root", str(RUN), "--workers", "2",
           "--batch", "2", "--test_batch", "4")


In [ ]:
import json
report = json.loads((RUN / "report.json").read_text())
print("Selected support samples:", report["support_samples"])
print("Best epoch:", report["best_epoch"], "score:", report["best_score"])
print(json.dumps(report["best_metrics"], indent=2))


Optional: read the saved best checkpoint and evaluate it again. The single report.json gains a reevaluation section; no new report file is created. The later --checkpoint argument overrides the base checkpoint in COMMON.


In [ ]:
# Optional re-evaluation:
# run_script("test.py", "--checkpoint", str(RUN / "checkpoints/best.pth"),
#            "--log_root", str(RUN), "--workers", "2", "--test_batch", "4")


Resume after attaching previous adapted output: copy the entire adapted run
folder from Input back to RUN, including report.json/checkpoint.pth/best.pth.
Keep the original base checkpoint in Input too. K, seed, batch, learning rate,
input file contents and base checkpoint must match the previous adapted run.
Leave the fresh-training cell disabled and uncomment the following cell.


In [ ]:
# Optional resume (after restoring RUN):
# run_script("train.py", "--resume", "--log_root", str(RUN),
#            "--workers", "2", "--batch", "2", "--test_batch", "4")


Outputs: checkpoints/best.pth, checkpoints/checkpoint.pth and one report.json containing selected support IDs, baseline, every epoch metrics and best predictions. Download/save them before ending the session.


In [ ]:
import shutil
archive = shutil.make_archive(f"/kaggle/working/gazeformer_s_k{K}_seed{SEED}_by_condition", "zip",
                             root_dir=RUN.parent, base_dir=RUN.name)
print(archive)
